# Mask R-CNN 학습

로컬 환경(data_wound_seg/, CPU/MPS) 기준으로 작성됨

## 1. 설정 | 경로,디바이스

In [ ]:
import os, glob
import numpy as np
import torch
import torchvision
from torchvision.transforms import functional as F

ROOT = "data_wound_seg"
TRAIN_IMG_DIR = os.path.join(ROOT, "train_images")
TRAIN_MSK_DIR = os.path.join(ROOT, "train_masks")
TEST_IMG_DIR  = os.path.join(ROOT, "test_images")

train_device = torch.device("cpu")  # 학습은 CPU 고정(안정성)
infer_device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")  # 추론은 MPS
print("train_device:", train_device, "| infer_device:", infer_device)
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)

train_device: cpu | infer_device: mps
torch: 2.10.0 torchvision: 0.25.0


## 2. Dataset

In [1]:
import cv2
from torch.utils.data import Dataset

class WoundSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, file_list=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        
        if file_list is None:
            self.files = sorted([os.path.basename(p) for p in glob.glob(os.path.join(img_dir, "*"))])
        else:
            self.files = file_list

        # 이미지/마스크 둘 다 존재하는 것만 유지
        valid = []
        for fn in self.files:
            if os.path.exists(os.path.join(img_dir, fn)) and os.path.exists(os.path.join(mask_dir, fn)):
                valid.append(fn)
        self.files = valid

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fn = self.files[idx]
        img_path = os.path.join(self.img_dir, fn)
        msk_path = os.path.join(self.mask_dir, fn)

        # 이미지: RGB float tensor [C,H,W] 0~1
        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (512, 512)) #이미지 사이즈 줄임 -> 시간 너무 오래 걸림 방지
        img = torch.from_numpy(img_rgb).permute(2,0,1).float() / 255.0

        # 마스크: 0/1 uint8 -> torch uint8 
        m = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(msk_path)
        mask = (m > 127).astype(np.uint8)
        mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST) #이미지 사이즈 줄임 -> 시간 너무 오래 걸림 방지
        
        '''
        # bbox 추출 (마스크가 비었으면 예외)
        ys, xs = np.where(mask > 0)
        if len(xs) == 0 or len(ys) == 0:
            # 비어있는 마스크는 학습에서 제외 -> 에러 처리
            raise ValueError(f"Empty mask: {msk_path}")
        '''
        
        ys, xs = np.where(mask > 0)
        if len(xs) == 0 or len(ys) == 0:
            # 빈 마스크면 다른 샘플로 교체
            new_idx = (idx + 1) % len(self.files)
            return self.__getitem__(new_idx)


        x1, x2 = xs.min(), xs.max()
        y1, y2 = ys.min(), ys.max()
        
        #target dict 구성
        boxes = torch.tensor([[x1, y1, x2, y2]], dtype=torch.float32)
        labels = torch.tensor([1], dtype=torch.int64)  
        masks  = torch.from_numpy(mask[None, :, :])    

        
        #COCO 포맷을 따르는 관례적인 키
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        iscrowd = torch.zeros((1,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd,
        }
        return img, target, fn

def collate_fn(batch):
    imgs, targets, fns = zip(*batch)
    return list(imgs), list(targets), list(fns)


## 3. DataLoader + 모델 생성

In [2]:
from torch.utils.data import DataLoader
'''
train_ds = WoundSegDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR)

#DataLoader
train_loader = DataLoader(
    train_ds,
    batch_size=1,      # 메모리 부족 시, 1로 바꿀 것
    shuffle=True,
    num_workers=0,     
    collate_fn=collate_fn
)
'''

train_ds = WoundSegDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR)

# 100장만으로 축소 (디버깅/파이프라인 확인용)
small_files = train_ds.files[:100]  
train_ds_small = WoundSegDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR, file_list=small_files)

# 3) DataLoader도 small dataset / batch_size=1
train_loader = DataLoader(
    train_ds_small,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

print("train_ds_small size:", len(train_ds_small))


# Torchvision Mask R-CNN 
#전이학습
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

#클래스 2개만 필요 (class 0: background,class 1: wound)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 2)
#마스크도 클래스 수(2) 만큼만 필요
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden = 256
model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
    in_features_mask, hidden, 2
)




NameError: name 'TRAIN_IMG_DIR' is not defined

## 4. 학습루프

In [ ]:
model = model.to(train_device)

import torch.optim as optim
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

print("model device (train):", next(model.parameters()).device)

#1 epoch 학습
def train_one_epoch(model, loader, optimizer):
    model.train() #학습모드전환
    total = 0.0 #손실 누적 변수
    
    for imgs, targets, _ in loader:
        imgs = [img.to(train_device) for img in imgs]
        targets = [{k: v.to(train_device) for k, v in t.items()} for t in targets]


        loss_dict = model(imgs, targets)  #Torchvision Mask R-CNN 학습
        loss = sum(loss for loss in loss_dict.values())

        #역전파 업데이트
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    return total / max(1, len(loader))
    


# 3 epoch 반복 -> 시간 너무 오래 걸려서 1로 변경
for epoch in range(1):  
    avg_loss = train_one_epoch(model, train_loader, optimizer)
    print(f"epoch {epoch+1} | avg_loss={avg_loss:.4f}")
    
    

model device (train): cpu
epoch 1 | avg_loss=0.6630


# Mask R-CNN 추론

## 1. 모델을 MPS로 옮김 && 추론 모드

In [6]:
import torch

infer_device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model = model.to(infer_device)
model.eval()

print("infer_device:", infer_device)
print("model device (infer):", next(model.parameters()).device)


infer_device: mps
model device (infer): mps:0


## 2. test_images 추론 / 오버레이 저장 / CSV 기록

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import time
import matplotlib.pyplot as plt


TEST_IMG_DIR = "data_wound_seg/test_images"
TEST_MSK_DIR = "data_wound_seg/test_masks"
OUT_DIR = "data_wound_seg/out/infer_test_50_final"

# 데모용 샘플 개수
NUM_SAMPLES = 50

# 임계값
score_thr = 0.5
mask_thr  = 0.5
USE_SCORE_THR = True

# 속도 옵션
RESIZE_TO_512 = True
SAVE_OVERLAY = True

# 폴더 생성
os.makedirs(OUT_DIR, exist_ok=True)

# plot 폴더 생성
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

def read_gt_bin(mask_path: str, target_size=None):
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if gt is None:
        return None
    if target_size is not None:
        gt = cv2.resize(gt, target_size, interpolation=cv2.INTER_NEAREST)
    return (gt > 0).astype(np.uint8)

def area_from_bin(bin_mask: np.ndarray) -> int:
    return int(bin_mask.sum())

def compute_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray):
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)

    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, np.logical_not(gt)).sum())
    fn = int(np.logical_and(np.logical_not(pred), gt).sum())

    union = tp + fp + fn
    iou = (tp / union) if union > 0 else None

    denom_dice = (2 * tp + fp + fn)
    dice = (2 * tp / denom_dice) if denom_dice > 0 else None

    prec_den = (tp + fp)
    rec_den = (tp + fn)
    precision = (tp / prec_den) if prec_den > 0 else None
    recall = (tp / rec_den) if rec_den > 0 else None

    return iou, dice, precision, recall, tp, fp, fn

def make_overlay(bgr, pred_bin, gt_bin, alpha=0.55):
    """
    GT only  : green
    Pred only: red
    Overlap  : yellow
    """
    h, w = bgr.shape[:2]
    overlay = bgr.copy()

    gt = (gt_bin == 1)
    pr = (pred_bin == 1)
    overlap = np.logical_and(gt, pr)
    gt_only = np.logical_and(gt, np.logical_not(pr))
    pr_only = np.logical_and(pr, np.logical_not(gt))

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[gt_only] = (0, 255, 0)
    color[pr_only] = (0, 0, 255)
    color[overlap] = (0, 255, 255)

    mask_any = (gt_only | pr_only | overlap)
    overlay[mask_any] = (overlay[mask_any] * (1 - alpha) + color[mask_any] * alpha).astype(np.uint8)
    return overlay


# 파일 list
fns_all = sorted([
    fn for fn in os.listdir(TEST_IMG_DIR)
    if fn.lower().endswith((".png", ".jpg", ".jpeg"))
])
print("num test images (all):", len(fns_all))

# 데모 sample 선택
areas = []
gt_bin_map = {}
gt_area_map = {}

target_size = (512, 512) if RESIZE_TO_512 else None

for fn in fns_all:
    gt_path = os.path.join(TEST_MSK_DIR, fn)
    if not os.path.exists(gt_path):
        continue
    gt_bin = read_gt_bin(gt_path, target_size=target_size)
    if gt_bin is None:
        continue
    a = area_from_bin(gt_bin)
    if a == 0:
        continue
    areas.append((fn, a))
    gt_bin_map[fn] = gt_bin
    gt_area_map[fn] = a

areas.sort(key=lambda x: x[1])
n = len(areas)
if n == 0:
    raise RuntimeError("GT mask(>0) 샘플이 0개입니다.")

# small/mid/large 이미지 선택
K = min(NUM_SAMPLES, n)
if n <= K:
    fns = [fn for fn, _ in areas]
else:
    k_small = int(round(K * 0.30))
    k_mid   = int(round(K * 0.40))
    k_large = K - k_small - k_mid

    small = [fn for fn, _ in areas[:k_small]]
    mid_start = max(0, n // 2 - k_mid // 2)
    mid = [fn for fn, _ in areas[mid_start:mid_start + k_mid]]
    large = [fn for fn, _ in areas[-k_large:]]

    seen = set()
    fns = []
    for x in (small + mid + large):
        if x not in seen:
            fns.append(x)
            seen.add(x)
    fns = fns[:K]

print("num test images (selected):", len(fns))


# Model 준비
model.eval()
model.to(infer_device)

# 추론 및 평가
rows = []
t0 = time.time()
overlay_saved = 0

for idx, fn in enumerate(fns, 1):
    img_path = os.path.join(TEST_IMG_DIR, fn)

    bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if bgr is None:
        print("[skip] cannot read:", img_path)
        continue

    if RESIZE_TO_512:
        bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    img = (
        torch.from_numpy(rgb)
        .permute(2, 0, 1)
        .contiguous()
        .float()
        .div(255.0)
        .to(infer_device)
    )

    pred_score = None
    pred_bin = np.zeros((bgr.shape[0], bgr.shape[1]), dtype=np.uint8)

    with torch.inference_mode():
        out = model([img])[0]

    if len(out.get("scores", [])) > 0 and len(out.get("masks", [])) > 0:
        pred_score = float(out["scores"][0].item())

        use_mask = True
        if USE_SCORE_THR and pred_score < score_thr:
            use_mask = False

        if use_mask:
            pm = out["masks"][0, 0].detach().cpu().numpy()
            pred_bin = (pm > mask_thr).astype(np.uint8)

    pred_area = int(pred_bin.sum())

    # GT
    gt_bin = gt_bin_map.get(fn, None)
    if gt_bin is None:
        gt_path = os.path.join(TEST_MSK_DIR, fn)
        gt_bin = read_gt_bin(gt_path, target_size=target_size)
    gt_area = int(gt_bin.sum()) if gt_bin is not None else None

    # metrics
    iou = dice = precision = recall = None
    tp = fp = fn_ = None
    abs_err = rel_err = None

    if gt_bin is not None and gt_area is not None and gt_area > 0:
        iou, dice, precision, recall, tp, fp, fn_ = compute_metrics(pred_bin, gt_bin)
        abs_err = abs(pred_area - gt_area)
        rel_err = abs_err / gt_area

    # overlay 저장 
    out_path = None
    if SAVE_OVERLAY and gt_bin is not None:
        overlay = make_overlay(bgr, pred_bin, gt_bin, alpha=0.55)
        out_path = os.path.join(OUT_DIR, f"overlay_{fn}")
        ok = cv2.imwrite(out_path, overlay)
        if ok:
            overlay_saved += 1
        else:
            print("[WARN] overlay save failed:", out_path)

    rows.append({
        "filename": fn,
        "img_path": img_path,
        "overlay_path": out_path,
        "pred_score_top1": pred_score,
        "pred_area_px": pred_area,
        "gt_area_px": gt_area,
        "abs_err_px": abs_err,
        "rel_err": rel_err,
        "iou": iou,
        "dice": dice,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn_
    })

    if idx % 10 == 0 or idx == len(fns):
        elapsed = time.time() - t0
        print(f"[{idx}/{len(fns)}] elapsed={elapsed:.1f}s | overlays_saved={overlay_saved}")


df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "infer_results.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("saved csv ->", csv_path)

valid = df.dropna(subset=["iou", "dice", "precision", "recall", "abs_err_px", "rel_err"])

summary = {
    "num_selected": int(len(df)),
    "num_valid_gt": int(len(valid)),
    "score_thr": score_thr,
    "mask_thr": mask_thr,
    "USE_SCORE_THR": USE_SCORE_THR,
    "RESIZE_TO_512": RESIZE_TO_512,
    "SAVE_OVERLAY": SAVE_OVERLAY,
    "overlays_saved": int(overlay_saved),
    "mean_iou": float(valid["iou"].mean()) if len(valid) else None,
    "mean_dice": float(valid["dice"].mean()) if len(valid) else None,
    "mean_precision": float(valid["precision"].mean()) if len(valid) else None,
    "mean_recall": float(valid["recall"].mean()) if len(valid) else None,
    "MAE_abs_err_px": float(valid["abs_err_px"].mean()) if len(valid) else None,
    "MAPE_rel_err": float(valid["rel_err"].mean()) if len(valid) else None,
    "pred_area_zero_rate": float((df["pred_area_px"] == 0).mean()) if len(df) else None,
}

summary_df = pd.DataFrame([summary])
summary_csv = os.path.join(OUT_DIR, "summary.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("saved summary ->", summary_csv)


# Plots
# 1) pred_area vs gt_area scatter
plt.figure()
sub = df.dropna(subset=["gt_area_px"])
plt.scatter(sub["gt_area_px"], sub["pred_area_px"])
plt.xlabel("GT area (px)")
plt.ylabel("Pred area (px)")
plt.title("Pred area vs GT area")
scatter_path = os.path.join(PLOT_DIR, "area_scatter.png")
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.close()

# 2) IoU histogram
plt.figure()
sub2 = df.dropna(subset=["iou"])
plt.hist(sub2["iou"].values, bins=20)
plt.xlabel("IoU")
plt.ylabel("Count")
plt.title("IoU histogram")
hist_path = os.path.join(PLOT_DIR, "iou_hist.png")
plt.savefig(hist_path, dpi=150, bbox_inches="tight")
plt.close()

print("saved plots ->", PLOT_DIR)
print("saved overlays ->", OUT_DIR)

df.head()


num test images (all): 539
num test images (selected): 50
[10/50] elapsed=701.2s | overlays_saved=10
[20/50] elapsed=1577.2s | overlays_saved=20
[30/50] elapsed=2228.7s | overlays_saved=30
[40/50] elapsed=3765.9s | overlays_saved=40
[50/50] elapsed=4390.6s | overlays_saved=50
saved csv -> data_wound_seg/out/infer_test_50_final/infer_results.csv
saved summary -> data_wound_seg/out/infer_test_50_final/summary.csv
saved plots -> data_wound_seg/out/infer_test_50_final/plots
saved overlays -> data_wound_seg/out/infer_test_50_final


,filename,img_path,overlay_path,pred_score_top1,pred_area_px,gt_area_px,abs_err_px,rel_err,iou,dice,precision,recall,tp,fp,fn
0,fusc_0816.png,data_wound_seg/test_images/fusc_0816.png,data_wound_seg/out/infer_test_50_final/overlay...,0.762161,307,20,287,14.350000,0.065147,0.122324,0.065147,1.000000,20,287,0
1,fusc_0071.png,data_wound_seg/test_images/fusc_0071.png,data_wound_seg/out/infer_test_50_final/overlay...,0.924956,1291,24,1267,52.791667,0.000000,0.000000,0.000000,0.000000,0,1291,24
2,fusc_0351.png,data_wound_seg/test_images/fusc_0351.png,data_wound_seg/out/infer_test_50_final/overlay...,0.885898,632,33,599,18.151515,0.052215,0.099248,0.052215,1.000000,33,599,0
3,fusc_0079.png,data_wound_seg/test_images/fusc_0079.png,data_wound_seg/out/infer_test_50_final/overlay...,0.553313,267,43,224,5.209302,0.156716,0.270968,0.157303,0.976744,42,225,1
4,fusc_0692.png,data_wound_seg/test_images/fusc_0692.png,data_wound_seg/out/infer_test_50_final/overlay...,0.948407,1280,45,1235,27.444444,0.000000,0.000000,0.000000,0.000000,0,1280,45


In [15]:
import os
import shutil
import pandas as pd

OUT_DIR = "data_wound_seg/out/infer_test_50_final"   
CSV_PATH = os.path.join(OUT_DIR, "infer_results.csv")

# 몇 개 뽑을지
TOPK = 10

# ====== 로드 ======
df = pd.read_csv(CSV_PATH)

# overlay_path가 비어있거나 파일이 없는 경우 걸러내기
def overlay_exists(p):
    return isinstance(p, str) and len(p) > 0 and os.path.exists(p)

df["overlay_ok"] = df["overlay_path"].apply(overlay_exists)
df_ok = df[df["overlay_ok"]].copy()

# 저장 폴더
CASE_DIR = os.path.join(OUT_DIR, "cases")
os.makedirs(CASE_DIR, exist_ok=True)

def copy_overlays(sub_df, dst_folder, reason_col=None):
    dst = os.path.join(CASE_DIR, dst_folder)
    os.makedirs(dst, exist_ok=True)

    copied = 0
    for _, row in sub_df.iterrows():
        src = row["overlay_path"]
        if not overlay_exists(src):
            continue

        fn = os.path.basename(src)  # overlay_fusc_xxxx.png
        # 파일명에 점수/이유를 붙여서 정렬해도 알아보기 쉽게
        prefix = ""
        if reason_col is not None and pd.notna(row.get(reason_col, None)):
            val = row[reason_col]
            try:
                prefix = f"{reason_col}-{float(val):.4f}__"
            except:
                prefix = f"{reason_col}-{val}__"

        dst_path = os.path.join(dst, prefix + fn)
        shutil.copy2(src, dst_path)
        copied += 1

    print(f"[{dst_folder}] copied: {copied} files -> {dst}")

# ====== 1) Best: IoU Top-K (GT가 있는 valid 케이스에서) ======
valid = df_ok.dropna(subset=["iou"]).copy()
best_iou = valid.sort_values("iou", ascending=False).head(TOPK)
copy_overlays(best_iou, f"best_iou_top{TOPK}", reason_col="iou")

# ====== 2) Worst: IoU Bottom-K (valid에서) ======
worst_iou = valid.sort_values("iou", ascending=True).head(TOPK)
copy_overlays(worst_iou, f"worst_iou_bottom{TOPK}", reason_col="iou")

# ====== 3) Worst: pred_area = 0 (완전 미검출) ======
worst_zero = df_ok[df_ok["pred_area_px"] == 0].copy()
# 너무 많으면 TOPK만
worst_zero = worst_zero.head(max(TOPK, len(worst_zero)))
copy_overlays(worst_zero, "worst_pred_zero", reason_col="pred_score_top1")

# ====== 4) Worst: 상대오차(rel_err) Top-K (valid에서) ======
valid_err = df_ok.dropna(subset=["rel_err"]).copy()
worst_rel = valid_err.sort_values("rel_err", ascending=False).head(TOPK)
copy_overlays(worst_rel, f"worst_relerr_top{TOPK}", reason_col="rel_err")

# ====== (옵션) Best: rel_err 작은 Top-K ======
best_rel = valid_err.sort_values("rel_err", ascending=True).head(TOPK)
copy_overlays(best_rel, f"best_relerr_small_top{TOPK}", reason_col="rel_err")

print("Done. Check:", CASE_DIR)


[best_iou_top10] copied: 10 files -> data_wound_seg/out/infer_test_50_final/cases/best_iou_top10
[worst_iou_bottom10] copied: 10 files -> data_wound_seg/out/infer_test_50_final/cases/worst_iou_bottom10
[worst_pred_zero] copied: 5 files -> data_wound_seg/out/infer_test_50_final/cases/worst_pred_zero
[worst_relerr_top10] copied: 10 files -> data_wound_seg/out/infer_test_50_final/cases/worst_relerr_top10
[best_relerr_small_top10] copied: 10 files -> data_wound_seg/out/infer_test_50_final/cases/best_relerr_small_top10
Done. Check: data_wound_seg/out/infer_test_50_final/cases


In [9]:
#개선 코드 
import os
import cv2
import numpy as np
import pandas as pd
import torch
import time
import matplotlib.pyplot as plt

TEST_IMG_DIR = "data_wound_seg/test_images"
TEST_MSK_DIR = "data_wound_seg/test_masks"
OUT_DIR = "data_wound_seg/out/infer_test_50_improve"

# 데모용 샘플 개수
NUM_SAMPLES = 50

# 임계값
score_thr = 0.5
mask_thr  = 0.5
USE_SCORE_THR = True

# 속도 옵션
RESIZE_TO_512 = True
SAVE_OVERLAY = True

os.makedirs(OUT_DIR, exist_ok=True)

PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

def read_gt_bin(mask_path: str, target_size=None):
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if gt is None:
        return None
    if target_size is not None:
        gt = cv2.resize(gt, target_size, interpolation=cv2.INTER_NEAREST)
    return (gt > 0).astype(np.uint8)

def area_from_bin(bin_mask: np.ndarray) -> int:
    return int(bin_mask.sum())

def compute_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray):
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)

    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, np.logical_not(gt)).sum())
    fn = int(np.logical_and(np.logical_not(pred), gt).sum())

    union = tp + fp + fn
    iou = (tp / union) if union > 0 else None

    denom_dice = (2 * tp + fp + fn)
    dice = (2 * tp / denom_dice) if denom_dice > 0 else None

    prec_den = (tp + fp)
    rec_den = (tp + fn)
    precision = (tp / prec_den) if prec_den > 0 else None
    recall = (tp / rec_den) if rec_den > 0 else None

    return iou, dice, precision, recall, tp, fp, fn

def make_overlay(bgr, pred_bin, gt_bin, alpha=0.55):
    """
    GT only  : green
    Pred only: red
    Overlap  : yellow
    """
    h, w = bgr.shape[:2]
    overlay = bgr.copy()

    gt = (gt_bin == 1)
    pr = (pred_bin == 1)
    overlap = np.logical_and(gt, pr)
    gt_only = np.logical_and(gt, np.logical_not(pr))
    pr_only = np.logical_and(pr, np.logical_not(gt))

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[gt_only] = (0, 255, 0)
    color[pr_only] = (0, 0, 255)
    color[overlap] = (0, 255, 255)

    mask_any = (gt_only | pr_only | overlap)
    overlay[mask_any] = (overlay[mask_any] * (1 - alpha) + color[mask_any] * alpha).astype(np.uint8)
    return overlay

#후처리 함수 
def postprocess_mask(bin_mask: np.ndarray,
                     do_close=True,
                     do_open=True,
                     k_close=7,
                     k_open=3,
                     min_area=0):
    m = (bin_mask.astype(np.uint8) * 255)

    if do_close:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)

    if do_open:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)

    m = (m > 0).astype(np.uint8)

    # 아주 작은 조각 제거
    if min_area > 0:
        num, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        out = np.zeros_like(m)
        for i in range(1, num):
            if stats[i, cv2.CC_STAT_AREA] >= min_area:
                out[labels == i] = 1
        m = out

    return m

# 파일 리스트
fns_all = sorted([
    fn for fn in os.listdir(TEST_IMG_DIR)
    if fn.lower().endswith((".png", ".jpg", ".jpeg"))
])
print("num test images (all):", len(fns_all))

# demo 샘플 선택
areas = []
gt_bin_map = {}
gt_area_map = {}

target_size = (512, 512) if RESIZE_TO_512 else None

for fn in fns_all:
    gt_path = os.path.join(TEST_MSK_DIR, fn)
    if not os.path.exists(gt_path):
        continue
    gt_bin = read_gt_bin(gt_path, target_size=target_size)
    if gt_bin is None:
        continue
    a = area_from_bin(gt_bin)
    if a == 0:
        continue
    areas.append((fn, a))
    gt_bin_map[fn] = gt_bin
    gt_area_map[fn] = a

areas.sort(key=lambda x: x[1])
n = len(areas)
if n == 0:
    raise RuntimeError("GT mask(>0) 샘플이 0개입니다.")

# small/mid/large 이미지 선택
K = min(NUM_SAMPLES, n)
if n <= K:
    fns = [fn for fn, _ in areas]
else:
    k_small = int(round(K * 0.30))
    k_mid   = int(round(K * 0.40))
    k_large = K - k_small - k_mid

    small = [fn for fn, _ in areas[:k_small]]
    mid_start = max(0, n // 2 - k_mid // 2)
    mid = [fn for fn, _ in areas[mid_start:mid_start + k_mid]]
    large = [fn for fn, _ in areas[-k_large:]]

    seen = set()
    fns = []
    for x in (small + mid + large):
        if x not in seen:
            fns.append(x)
            seen.add(x)
    fns = fns[:K]

print("num test images (selected):", len(fns))

# Model 준비
model.eval()
model.to(infer_device)

# Inference + Evaluation
rows = []
t0 = time.time()
overlay_saved = 0

for idx, fn in enumerate(fns, 1):
    img_path = os.path.join(TEST_IMG_DIR, fn)

    bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if bgr is None:
        print("[skip] cannot read:", img_path)
        continue

    if RESIZE_TO_512:
        bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    img = (
        torch.from_numpy(rgb)
        .permute(2, 0, 1)
        .contiguous()
        .float()
        .div(255.0)
        .to(infer_device)
    )

    pred_score = None
    pred_bin = np.zeros((bgr.shape[0], bgr.shape[1]), dtype=np.uint8)

    with torch.inference_mode():
        out = model([img])[0]

    # 개선: 여러 mask union 
    scores = out.get("scores", None)
    masks  = out.get("masks", None)

    if scores is not None and masks is not None and len(scores) > 0 and len(masks) > 0:
        scores_np = scores.detach().cpu().numpy()
        pred_score = float(scores_np[0])  # top1 score (기록용)

        # score_thr 이상만 사용 (USE_SCORE_THR=False면 전부 사용)
        keep = (scores_np >= score_thr) if USE_SCORE_THR else np.ones_like(scores_np, dtype=bool)

        # (옵션) topK 제한 원하면 주석 해제
        # topK = 3
        # keep = keep & (np.arange(len(scores_np)) < topK)

        if keep.any():
            pm = masks[keep, 0].detach().cpu().numpy()  # [K, H, W]
            pred_bin = (pm.max(axis=0) > mask_thr).astype(np.uint8)  # union(OR)

            # 후처리 (구멍 메우기/노이즈 제거/작은 조각 제거)
            pred_bin = postprocess_mask(pred_bin, k_close=7, k_open=3, min_area=200)

    # pred_area는 pred_bin 만든 다음 계산
    pred_area = int(pred_bin.sum())


    # GT
    gt_bin = gt_bin_map.get(fn, None)
    if gt_bin is None:
        gt_path = os.path.join(TEST_MSK_DIR, fn)
        gt_bin = read_gt_bin(gt_path, target_size=target_size)
    gt_area = int(gt_bin.sum()) if gt_bin is not None else None

    # metrics
    iou = dice = precision = recall = None
    tp = fp = fn_ = None
    abs_err = rel_err = None

    if gt_bin is not None and gt_area is not None and gt_area > 0:
        iou, dice, precision, recall, tp, fp, fn_ = compute_metrics(pred_bin, gt_bin)
        abs_err = abs(pred_area - gt_area)
        rel_err = abs_err / gt_area

    # overlay 저장
    out_path = None
    if SAVE_OVERLAY and gt_bin is not None:
        overlay = make_overlay(bgr, pred_bin, gt_bin, alpha=0.55)
        out_path = os.path.join(OUT_DIR, f"overlay_{fn}")
        ok = cv2.imwrite(out_path, overlay)
        if ok:
            overlay_saved += 1
        else:
            print("[WARN] overlay save failed:", out_path)

    rows.append({
        "filename": fn,
        "img_path": img_path,
        "overlay_path": out_path,
        "pred_score_top1": pred_score,
        "pred_area_px": pred_area,
        "gt_area_px": gt_area,
        "abs_err_px": abs_err,
        "rel_err": rel_err,
        "iou": iou,
        "dice": dice,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn_
    })

    if idx % 10 == 0 or idx == len(fns):
        elapsed = time.time() - t0
        print(f"[{idx}/{len(fns)}] elapsed={elapsed:.1f}s | overlays_saved={overlay_saved}")


df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "infer_results.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("saved csv ->", csv_path)

valid = df.dropna(subset=["iou", "dice", "precision", "recall", "abs_err_px", "rel_err"])

summary = {
    "num_selected": int(len(df)),
    "num_valid_gt": int(len(valid)),
    "score_thr": score_thr,
    "mask_thr": mask_thr,
    "USE_SCORE_THR": USE_SCORE_THR,
    "RESIZE_TO_512": RESIZE_TO_512,
    "SAVE_OVERLAY": SAVE_OVERLAY,
    "overlays_saved": int(overlay_saved),
    "mean_iou": float(valid["iou"].mean()) if len(valid) else None,
    "mean_dice": float(valid["dice"].mean()) if len(valid) else None,
    "mean_precision": float(valid["precision"].mean()) if len(valid) else None,
    "mean_recall": float(valid["recall"].mean()) if len(valid) else None,
    "MAE_abs_err_px": float(valid["abs_err_px"].mean()) if len(valid) else None,
    "MAPE_rel_err": float(valid["rel_err"].mean()) if len(valid) else None,
    "pred_area_zero_rate": float((df["pred_area_px"] == 0).mean()) if len(df) else None,
}

summary_df = pd.DataFrame([summary])
summary_csv = os.path.join(OUT_DIR, "summary.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("saved summary ->", summary_csv)


# Plots
# 1) pred_area vs gt_area scatter
plt.figure()
sub = df.dropna(subset=["gt_area_px"])
plt.scatter(sub["gt_area_px"], sub["pred_area_px"])
plt.xlabel("GT area (px)")
plt.ylabel("Pred area (px)")
plt.title("Pred area vs GT area")
scatter_path = os.path.join(PLOT_DIR, "area_scatter.png")
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.close()

# 2) IoU histogram
plt.figure()
sub2 = df.dropna(subset=["iou"])
plt.hist(sub2["iou"].values, bins=20)
plt.xlabel("IoU")
plt.ylabel("Count")
plt.title("IoU histogram")
hist_path = os.path.join(PLOT_DIR, "iou_hist.png")
plt.savefig(hist_path, dpi=150, bbox_inches="tight")
plt.close()

print("saved plots ->", PLOT_DIR)
print("saved overlays ->", OUT_DIR)

df.head()


num test images (all): 539
num test images (selected): 50


KeyboardInterrupt: 

In [7]:
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_improved_mask_rcnn(num_classes):
    # 1. Pretrained Backbone: ResNet-50 FPN (Feature Pyramid Network) 활용
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)

    # 2. Anchor Generator 최적화: 상처(Wound)의 크기가 다양하므로 작은 앵커 사이즈 추가
    # 기본 (32, 64, 128, 256, 512)에서 (16,)을 추가하여 미세한 영역 검출 성능 향상
    anchor_sizes = ((16,), (32,), (64,), (128,), (256,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    model.rpn.anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)

    # 3. Box Predictor 교체: 새로운 클래스 수에 맞게 조정
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # 4. Mask Predictor 교체: Segmentation 정밀도 향상
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

# 성능 향상을 위한 추가 팁:
# - Data Augmentation: Albumentations 라이브러리를 사용해 Flip, Rotate, Brightness, Contrast 조절
# - Learning Rate Scheduler: torch.optim.lr_scheduler.CosineAnnealingLR 등을 사용하여 수렴 성능 개선
# - Image Resizing: 입력 이미지를 800px 이상으로 유지하여 세부 특징 보존
import os
import cv2
import numpy as np
import pandas as pd
import torch
import time
import matplotlib.pyplot as plt

TEST_IMG_DIR = "data_wound_seg/test_images"
TEST_MSK_DIR = "data_wound_seg/test_masks"
OUT_DIR = "data_wound_seg/out/infer_test_50_improve_3"

# 데모용 샘플 개수
NUM_SAMPLES = 50

# 임계값 
score_thr = 0.3 #변경 0.5 -> 0.3
mask_thr  = 0.5
USE_SCORE_THR = True

# 속도 옵션
RESIZE_TO_512 = True
SAVE_OVERLAY = True

os.makedirs(OUT_DIR, exist_ok=True)

PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

def read_gt_bin(mask_path: str, target_size=None):
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if gt is None:
        return None
    if target_size is not None:
        gt = cv2.resize(gt, target_size, interpolation=cv2.INTER_NEAREST)
    return (gt > 0).astype(np.uint8)

def area_from_bin(bin_mask: np.ndarray) -> int:
    return int(bin_mask.sum())

def compute_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray):
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)

    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, np.logical_not(gt)).sum())
    fn = int(np.logical_and(np.logical_not(pred), gt).sum())

    union = tp + fp + fn
    iou = (tp / union) if union > 0 else None

    denom_dice = (2 * tp + fp + fn)
    dice = (2 * tp / denom_dice) if denom_dice > 0 else None

    prec_den = (tp + fp)
    rec_den = (tp + fn)
    precision = (tp / prec_den) if prec_den > 0 else None
    recall = (tp / rec_den) if rec_den > 0 else None

    return iou, dice, precision, recall, tp, fp, fn

def make_overlay(bgr, pred_bin, gt_bin, alpha=0.55):
    """
    GT only  : green
    Pred only: red
    Overlap  : yellow
    """
    h, w = bgr.shape[:2]
    overlay = bgr.copy()

    gt = (gt_bin == 1)
    pr = (pred_bin == 1)
    overlap = np.logical_and(gt, pr)
    gt_only = np.logical_and(gt, np.logical_not(pr))
    pr_only = np.logical_and(pr, np.logical_not(gt))

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[gt_only] = (0, 255, 0)
    color[pr_only] = (0, 0, 255)
    color[overlap] = (0, 255, 255)

    mask_any = (gt_only | pr_only | overlap)
    overlay[mask_any] = (overlay[mask_any] * (1 - alpha) + color[mask_any] * alpha).astype(np.uint8)
    return overlay

#후처리 함수 
def postprocess_mask(bin_mask: np.ndarray,
                     do_close=True,
                     do_open=True,
                     k_close=7,
                     k_open=3,
                     min_area=0):
    m = (bin_mask.astype(np.uint8) * 255)

    if do_close:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)

    if do_open:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)

    m = (m > 0).astype(np.uint8)

    # 아주 작은 조각 제거
    if min_area > 0:
        num, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        out = np.zeros_like(m)
        for i in range(1, num):
            if stats[i, cv2.CC_STAT_AREA] >= min_area:
                out[labels == i] = 1
        m = out

    return m

# 파일 리스트
fns_all = sorted([
    fn for fn in os.listdir(TEST_IMG_DIR)
    if fn.lower().endswith((".png", ".jpg", ".jpeg"))
])
print("num test images (all):", len(fns_all))

# demo 샘플 선택
areas = []
gt_bin_map = {}
gt_area_map = {}

target_size = (512, 512) if RESIZE_TO_512 else None

for fn in fns_all:
    gt_path = os.path.join(TEST_MSK_DIR, fn)
    if not os.path.exists(gt_path):
        continue
    gt_bin = read_gt_bin(gt_path, target_size=target_size)
    if gt_bin is None:
        continue
    a = area_from_bin(gt_bin)
    if a == 0:
        continue
    areas.append((fn, a))
    gt_bin_map[fn] = gt_bin
    gt_area_map[fn] = a

areas.sort(key=lambda x: x[1])
n = len(areas)
if n == 0:
    raise RuntimeError("GT mask(>0) 샘플이 0개입니다.")

# small/mid/large 이미지 선택
K = min(NUM_SAMPLES, n)
if n <= K:
    fns = [fn for fn, _ in areas]
else:
    k_small = int(round(K * 0.30))
    k_mid   = int(round(K * 0.40))
    k_large = K - k_small - k_mid

    small = [fn for fn, _ in areas[:k_small]]
    mid_start = max(0, n // 2 - k_mid // 2)
    mid = [fn for fn, _ in areas[mid_start:mid_start + k_mid]]
    large = [fn for fn, _ in areas[-k_large:]]

    seen = set()
    fns = []
    for x in (small + mid + large):
        if x not in seen:
            fns.append(x)
            seen.add(x)
    fns = fns[:K]

print("num test images (selected):", len(fns))

# Model 준비
model.eval()
model.to(infer_device)

# Inference + Evaluation
rows = []
t0 = time.time()
overlay_saved = 0

for idx, fn in enumerate(fns, 1):
    img_path = os.path.join(TEST_IMG_DIR, fn)

    bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if bgr is None:
        print("[skip] cannot read:", img_path)
        continue

    if RESIZE_TO_512:
        bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    img = (
        torch.from_numpy(rgb)
        .permute(2, 0, 1)
        .contiguous()
        .float()
        .div(255.0)
        .to(infer_device)
    )

    pred_score = None
    pred_bin = np.zeros((bgr.shape[0], bgr.shape[1]), dtype=np.uint8)

    with torch.inference_mode():
        out = model([img])[0]

    # 개선: 여러 mask union 
    scores = out.get("scores", None)
    masks  = out.get("masks", None)

    if scores is not None and masks is not None and len(scores) > 0 and len(masks) > 0:
        scores_np = scores.detach().cpu().numpy()
        pred_score = float(scores_np[0])  # top1 score (기록용)

        # score_thr 이상만 사용 (USE_SCORE_THR=False면 전부 사용)
        keep = (scores_np >= score_thr) if USE_SCORE_THR else np.ones_like(scores_np, dtype=bool)

        #topK는 3개로 고정
        topK = 3
        keep = keep & (np.arange(len(scores_np)) < topK)

        if keep.any():
            pm = masks[keep, 0].detach().cpu().numpy()  # [K, H, W]
            pred_bin = (pm.max(axis=0) > mask_thr).astype(np.uint8)  # union(OR)

            # 후처리 (구멍 메우기/노이즈 제거/작은 조각 제거)
            pred_bin = postprocess_mask(pred_bin, k_close=7, k_open=3, min_area=200)

    # pred_area는 pred_bin 만든 다음 계산
    pred_area = int(pred_bin.sum())


    # GT
    gt_bin = gt_bin_map.get(fn, None)
    if gt_bin is None:
        gt_path = os.path.join(TEST_MSK_DIR, fn)
        gt_bin = read_gt_bin(gt_path, target_size=target_size)
    gt_area = int(gt_bin.sum()) if gt_bin is not None else None

    # metrics
    iou = dice = precision = recall = None
    tp = fp = fn_ = None
    abs_err = rel_err = None

    if gt_bin is not None and gt_area is not None and gt_area > 0:
        iou, dice, precision, recall, tp, fp, fn_ = compute_metrics(pred_bin, gt_bin)
        abs_err = abs(pred_area - gt_area)
        rel_err = abs_err / gt_area

    # overlay 저장
    out_path = None
    if SAVE_OVERLAY and gt_bin is not None:
        overlay = make_overlay(bgr, pred_bin, gt_bin, alpha=0.55)
        out_path = os.path.join(OUT_DIR, f"overlay_{fn}")
        ok = cv2.imwrite(out_path, overlay)
        if ok:
            overlay_saved += 1
        else:
            print("[WARN] overlay save failed:", out_path)

    rows.append({
        "filename": fn,
        "img_path": img_path,
        "overlay_path": out_path,
        "pred_score_top1": pred_score,
        "pred_area_px": pred_area,
        "gt_area_px": gt_area,
        "abs_err_px": abs_err,
        "rel_err": rel_err,
        "iou": iou,
        "dice": dice,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn_
    })

    if idx % 10 == 0 or idx == len(fns):
        elapsed = time.time() - t0
        print(f"[{idx}/{len(fns)}] elapsed={elapsed:.1f}s | overlays_saved={overlay_saved}")


df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "infer_results.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("saved csv ->", csv_path)

valid = df.dropna(subset=["iou", "dice", "precision", "recall", "abs_err_px", "rel_err"])

summary = {
    "num_selected": int(len(df)),
    "num_valid_gt": int(len(valid)),
    "score_thr": score_thr,
    "mask_thr": mask_thr,
    "USE_SCORE_THR": USE_SCORE_THR,
    "RESIZE_TO_512": RESIZE_TO_512,
    "SAVE_OVERLAY": SAVE_OVERLAY,
    "overlays_saved": int(overlay_saved),
    "mean_iou": float(valid["iou"].mean()) if len(valid) else None,
    "mean_dice": float(valid["dice"].mean()) if len(valid) else None,
    "mean_precision": float(valid["precision"].mean()) if len(valid) else None,
    "mean_recall": float(valid["recall"].mean()) if len(valid) else None,
    "MAE_abs_err_px": float(valid["abs_err_px"].mean()) if len(valid) else None,
    "MAPE_rel_err": float(valid["rel_err"].mean()) if len(valid) else None,
    "pred_area_zero_rate": float((df["pred_area_px"] == 0).mean()) if len(df) else None,
}

summary_df = pd.DataFrame([summary])
summary_csv = os.path.join(OUT_DIR, "summary.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("saved summary ->", summary_csv)


# Plots
# 1) pred_area vs gt_area scatter
plt.figure()
sub = df.dropna(subset=["gt_area_px"])
plt.scatter(sub["gt_area_px"], sub["pred_area_px"])
plt.xlabel("GT area (px)")
plt.ylabel("Pred area (px)")
plt.title("Pred area vs GT area")
scatter_path = os.path.join(PLOT_DIR, "area_scatter.png")
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.close()

# 2) IoU histogram
plt.figure()
sub2 = df.dropna(subset=["iou"])
plt.hist(sub2["iou"].values, bins=20)
plt.xlabel("IoU")
plt.ylabel("Count")
plt.title("IoU histogram")
hist_path = os.path.join(PLOT_DIR, "iou_hist.png")
plt.savefig(hist_path, dpi=150, bbox_inches="tight")
plt.close()

print("saved plots ->", PLOT_DIR)
print("saved overlays ->", OUT_DIR)

df.head()


num test images (all): 539
num test images (selected): 50
[10/50] elapsed=585.8s | overlays_saved=10
[20/50] elapsed=1179.5s | overlays_saved=20
[30/50] elapsed=1768.4s | overlays_saved=30
[40/50] elapsed=2365.4s | overlays_saved=40
[50/50] elapsed=2955.3s | overlays_saved=50
saved csv -> data_wound_seg/out/infer_test_50_improve_3/infer_results.csv
saved summary -> data_wound_seg/out/infer_test_50_improve_3/summary.csv
saved plots -> data_wound_seg/out/infer_test_50_improve_3/plots
saved overlays -> data_wound_seg/out/infer_test_50_improve_3


,filename,img_path,overlay_path,pred_score_top1,pred_area_px,gt_area_px,abs_err_px,rel_err,iou,dice,precision,recall,tp,fp,fn
0,fusc_0816.png,data_wound_seg/test_images/fusc_0816.png,data_wound_seg/out/infer_test_50_improve_3/ove...,0.451098,541,20,521,26.050000,0.036969,0.071301,0.036969,1.000000,20,521,0
1,fusc_0071.png,data_wound_seg/test_images/fusc_0071.png,data_wound_seg/out/infer_test_50_improve_3/ove...,0.723797,6124,24,6100,254.166667,0.003919,0.007807,0.003919,1.000000,24,6100,0
2,fusc_0351.png,data_wound_seg/test_images/fusc_0351.png,data_wound_seg/out/infer_test_50_improve_3/ove...,0.816023,1568,33,1535,46.515152,0.021046,0.041224,0.021046,1.000000,33,1535,0
3,fusc_0079.png,data_wound_seg/test_images/fusc_0079.png,data_wound_seg/out/infer_test_50_improve_3/ove...,0.656276,5406,43,5363,124.720930,0.007954,0.015783,0.007954,1.000000,43,5363,0
4,fusc_0692.png,data_wound_seg/test_images/fusc_0692.png,data_wound_seg/out/infer_test_50_improve_3/ove...,0.526150,4146,45,4101,91.133333,0.010610,0.020997,0.010613,0.977778,44,4102,1
